# 🔬 Notebook 02 — Feature Selection: Which Sensors Predict Failure?
## Fab Intelligence: Semiconductor Manufacturing Analytics

**Business Question:** *Out of 590 process sensors, which ones actually matter for predicting wafer failure — and can we reduce inspection scope to just those signals?*

**Why it matters:** A fab engineer can't monitor 590 sensors manually. Identifying the top 10–20 **critical process control points** is how you build a scalable early warning system.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

BLUE = '#1A3C6E'; RED = '#C0392B'; GREEN = '#27AE60'; ORANGE = '#E67E22'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': '#F8FAFC'})

df = pd.read_csv('../data/secom_combined.csv')
sensor_cols = [c for c in df.columns if c not in ['label', 'label_text']]
X_raw = df[sensor_cols]
y = (df['label'] == 1).astype(int)  # 1 = FAIL

print(f'Shape: {X_raw.shape}   |   Failures: {y.sum()} ({y.mean()*100:.1f}%)')

In [ ]:
# ── Step 1: Remove zero-variance sensors ──────────────────────────────────
std = X_raw.std()
zero_var_cols = std[std == 0].index.tolist()
X_filtered = X_raw.drop(columns=zero_var_cols)
print(f'Removed {len(zero_var_cols)} zero-variance sensors → {X_filtered.shape[1]} remain')

# ── Step 2: Impute missing values (median strategy) ───────────────────────
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X_filtered),
                          columns=X_filtered.columns)
print(f'Missing values after imputation: {X_imputed.isna().sum().sum()}')

In [ ]:
# ── Step 3: T-test — sensors that differ significantly between PASS/FAIL ──
print('Running t-tests across all sensors...')
ttest_results = []
for col in X_imputed.columns:
    pass_vals = X_imputed.loc[y == 0, col]
    fail_vals = X_imputed.loc[y == 1, col]
    t_stat, p_val = stats.ttest_ind(pass_vals, fail_vals, equal_var=False)
    effect_size = (fail_vals.mean() - pass_vals.mean()) / (X_imputed[col].std() + 1e-9)
    ttest_results.append({'sensor': col, 't_stat': t_stat, 'p_value': p_val,
                           'effect_size': abs(effect_size),
                           'pass_mean': pass_vals.mean(), 'fail_mean': fail_vals.mean()})

ttest_df = pd.DataFrame(ttest_results).sort_values('p_value')
significant = ttest_df[ttest_df['p_value'] < 0.05]
print(f'Sensors significant at p<0.05: {len(significant)} out of {len(ttest_df)}')
print('\nTop 15 most significant sensors:')
print(significant.head(15)[['sensor','p_value','effect_size','pass_mean','fail_mean']].to_string(index=False))

In [ ]:
# ── Step 4: Random Forest Feature Importance ──────────────────────────────
print('Training Random Forest for feature importance...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_scaled, y)

importance_df = pd.DataFrame({
    'sensor': X_imputed.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

top20 = importance_df.head(20)
print('\nTop 20 most important sensors (Random Forest):')
print(top20.to_string(index=False))

In [ ]:
# ── Figure 2: Feature Importance + T-test Combined ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor='#F8FAFC')
fig.suptitle('Critical Process Control Sensors — Failure Predictors', 
             fontsize=16, fontweight='bold', color=BLUE)

# Left: RF Importance
ax = axes[0]
colors = [RED if i < 5 else BLUE if i < 10 else GRAY 
          for i, _ in enumerate(top20['sensor'])]
bars = ax.barh(range(len(top20)), top20['importance'], color=colors, alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f'Sensor {s}' for s in top20['sensor']], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance Score')
ax.set_title('Top 20 Sensors by Random Forest\nFeature Importance', 
             fontweight='bold', color=BLUE)
ax.axvline(top20['importance'].mean(), color=ORANGE, linestyle='--',
           label=f'Mean importance: {top20["importance"].mean():.4f}')
# Annotations
for i, (_, row) in enumerate(top20.head(5).iterrows()):
    ax.text(row['importance'] + 0.0002, i, f'  ⚠ Critical', 
            va='center', fontsize=8, color=RED, fontweight='bold')
ax.legend(fontsize=9)

# Right: Effect size (PASS vs FAIL mean difference)
ax2 = axes[1]
top15_sig = significant.head(15).copy()
top15_sig['pct_diff'] = ((top15_sig['fail_mean'] - top15_sig['pass_mean']) 
                          / (top15_sig['pass_mean'].abs() + 1e-9)) * 100

colors2 = [RED if v > 0 else GREEN for v in top15_sig['pct_diff']]
ax2.barh(range(len(top15_sig)), top15_sig['pct_diff'], 
         color=colors2, alpha=0.85)
ax2.set_yticks(range(len(top15_sig)))
ax2.set_yticklabels([f'Sensor {s}' for s in top15_sig['sensor']], fontsize=9)
ax2.invert_yaxis()
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('% Difference (FAIL mean vs PASS mean)')
ax2.set_title('How Much Sensor Readings Differ\nBetween PASS and FAIL Wafers',
              fontweight='bold', color=BLUE)

plt.tight_layout()
plt.savefig('../docs/fig02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 02 saved.')

In [ ]:
# ── Business Summary ───────────────────────────────────────────────────────
top5 = top20.head(5)['sensor'].tolist()
top10_sensors = top20.head(10)['sensor'].tolist()
coverage = importance_df[importance_df['sensor'].isin(top10_sensors)]['importance'].sum()

print('='*55)
print('FEATURE SELECTION — BUSINESS SUMMARY')
print('='*55)
print(f'Total sensors analyzed   : {len(X_imputed.columns)}')
print(f'Significant sensors      : {len(significant)} (p < 0.05)')
print(f'Recommended monitoring   : Top 10 sensors')
print(f'Importance coverage      : {coverage*100:.1f}% of total predictive signal')
print()
print('TOP 5 CRITICAL SENSORS (highest failure correlation):')
for i, s in enumerate(top5, 1):
    imp = top20[top20['sensor']==s]['importance'].values[0]
    print(f'  {i}. Sensor {s:>4}  —  importance: {imp:.4f}')
print()
print('→ RECOMMENDATION: Focus SPC (Statistical Process Control)')
print('  on these 10 sensors. Reduces monitoring scope by 98.3%')
print('  while retaining majority of failure predictive power.')

# Save top sensors for next notebook
top20.to_csv('../data/top_sensors.csv', index=False)
print('\nTop sensors saved to data/top_sensors.csv')